# 01 · Extracción y consolidación de datos

**Objetivo:** extraer todos los estudios almacenados en la base de datos SQLite de Optuna y consolidarlos en un único DataFrame que sirva como fuente de datos para todos los análisis posteriores del TFM.

**Salidas de este notebook:**

| Fichero | Granularidad | Usado en |
|---|---|---|
| `master_results.csv` | Un `trial` por fila (~7000 filas) | `03_estadistica.ipynb` (no directamente), `04_convergencia.ipynb` |
| `study_summary.csv` | Un estudio por fila (140 filas) | `02_ranking_global.ipynb` |

`02_ranking_global.ipynb` normaliza y enriquece `study_summary.csv`, generando a su vez `study_summary_normalized.csv`, que es el fichero que consumen `03_estadistica.ipynb` y el resto de notebooks posteriores del TFM.


## Imports

In [1]:
import optuna
import pandas as pd
from pathlib import Path

d:\tfm\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuración

In [2]:
STORAGE = "sqlite:///tfm_final.db"

SAMPLERS = [
    "tpe",
    "cmaes",
    "random",
    "nsgaii",
    "nsgaiii",
    "qmc",
    "gp",
]

PROBLEMS = [f"zcat{i}" for i in range(1, 21)]

SEED_SUFFIX = "s3"  # sufijo de semilla usado al lanzar los estudios

OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Samplers ({len(SAMPLERS)}): {SAMPLERS}")
print(f"Problemas ({len(PROBLEMS)}): {PROBLEMS[0]}..{PROBLEMS[-1]}")
print(f"Estudios esperados: {len(SAMPLERS) * len(PROBLEMS)}")

Samplers (7): ['tpe', 'cmaes', 'random', 'nsgaii', 'nsgaiii', 'qmc', 'gp']
Problemas (20): zcat1..zcat20
Estudios esperados: 140


## Función de carga de un estudio

In [3]:
def load_study_dataframe(study_name: str) -> pd.DataFrame:
    """Carga un estudio de Optuna desde la base SQLite y lo devuelve como DataFrame,
    con una fila por trial."""
    study = optuna.load_study(
        study_name=study_name,
        storage=STORAGE,
    )

    df = study.trials_dataframe(
        attrs=(
            "number",
            "value",
            "datetime_start",
            "datetime_complete",
            "duration",
            "state",
            "params",
            "user_attrs",
            "system_attrs",
        )
    )

    return df

## Extracción de los 140 estudios

Se recorren los 7 *samplers* × 20 problemas ZCAT. El nombre de cada estudio en la base de datos sigue el patrón `{sampler}_{problem}_{seed}`.


In [4]:
dfs = []
n_loaded = 0
n_expected = len(SAMPLERS) * len(PROBLEMS)

for sampler in SAMPLERS:
    for problem in PROBLEMS:
        study_name = f"{sampler}_{problem}_{SEED_SUFFIX}"

        df = load_study_dataframe(study_name)
        df["sampler"] = sampler
        df["problem"] = problem
        dfs.append(df)

        n_loaded += 1
        if n_loaded % 20 == 0 or n_loaded == n_expected:
            print(f"  {n_loaded}/{n_expected} estudios cargados "
                  f"(último: {study_name})")

print(f"\nExtracción completada: {n_loaded} estudios cargados.")

  20/140 estudios cargados (último: tpe_zcat20_s3)
  40/140 estudios cargados (último: cmaes_zcat20_s3)
  60/140 estudios cargados (último: random_zcat20_s3)
  80/140 estudios cargados (último: nsgaii_zcat20_s3)
  100/140 estudios cargados (último: nsgaiii_zcat20_s3)
  120/140 estudios cargados (último: qmc_zcat20_s3)
  140/140 estudios cargados (último: gp_zcat20_s3)

Extracción completada: 140 estudios cargados.


## Consolidación en un DataFrame maestro

In [5]:
master_df = pd.concat(dfs, ignore_index=True)

# Reordenar columnas: identificadores primero, resto de columnas Optuna después
first_columns = ["sampler", "problem", "number", "value", "state", "duration"]
remaining = [c for c in master_df.columns if c not in first_columns]
master_df = master_df[first_columns + remaining]

# Orden cronológico dentro de cada experimento; facilita el análisis de convergencia posterior
master_df = master_df.sort_values(["sampler", "problem", "number"]).reset_index(drop=True)

master_df.head()

,sampler,problem,number,value,state,duration,datetime_start,datetime_complete,params_archive_prune_policy,params_archive_unbounded,...,system_attrs_cma:optimizer:3,system_attrs_cma:optimizer:4,system_attrs_cma:optimizer:5,system_attrs_cma:optimizer:6,system_attrs_cma:optimizer:7,system_attrs_cma:optimizer:8,system_attrs_cma:optimizer:9,system_attrs_NSGAIISampler:generation,system_attrs_NSGAIIISampler:generation,system_attrs_gp:relative_params:0
0,cmaes,zcat1,0,0.183527,COMPLETE,0 days 00:00:39.033047,2026-05-13 16:23:28.920672,2026-05-13 16:24:07.953719,crowding,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,cmaes,zcat1,1,0.254027,COMPLETE,0 days 00:00:42.255058,2026-05-13 16:23:28.927148,2026-05-13 16:24:11.182206,ref_dirs,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,cmaes,zcat1,2,0.166690,COMPLETE,0 days 00:01:01.929140,2026-05-13 16:23:28.940678,2026-05-13 16:24:30.869818,crowding,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,cmaes,zcat1,3,0.070692,COMPLETE,0 days 00:00:44.004137,2026-05-13 16:23:28.924053,2026-05-13 16:24:12.928190,mc_hv,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,cmaes,zcat1,4,0.000000,COMPLETE,0 days 00:00:45.572354,2026-05-13 16:24:08.016796,2026-05-13 16:24:53.589150,ref_dirs,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Guardado del dataset maestro

In [6]:
master_df.to_csv(OUTPUT_DIR / "master_results.csv", index=False)
print(f"Guardado: {(OUTPUT_DIR / 'master_results.csv').resolve()}")
print(f"Filas: {len(master_df):,}  |  Columnas: {master_df.shape[1]}")

Guardado: D:\tfm\tfm\data\processed\master_results.csv
Filas: 7,004  |  Columnas: 60


## Comprobaciones de integridad

Se espera: 7 *samplers* × 20 problemas = 140 combinaciones, con 50 *trials* cada una (7000 filas en total).


In [7]:
trials_per_study = master_df.groupby(["sampler", "problem"]).size()

n_studies = trials_per_study.shape[0]
n_samplers = master_df["sampler"].nunique()
n_problems = master_df["problem"].nunique()
trials_ok = (trials_per_study == trials_per_study.iloc[0]).all()

print(f"Total de filas:        {len(master_df):,}")
print(f"Estudios (sampler x problem): {n_studies}  (esperado: {n_samplers * n_problems})")
print(f"Samplers distintos:    {n_samplers}")
print(f"Problemas distintos:   {n_problems}")
print(f"Trials por estudio homogéneos: {trials_ok} "
      f"(min={trials_per_study.min()}, max={trials_per_study.max()})")


Total de filas:        7,004
Estudios (sampler x problem): 140  (esperado: 140)
Samplers distintos:    7
Problemas distintos:   20
Trials por estudio homogéneos: False (min=50, max=54)


## Resumen por estudio (agregado)

In [8]:
study_summary = (
    master_df
    .groupby(["sampler", "problem"])
    .agg(
        n_trials=("number", "count"),
        best_value=("value", "max"),
        mean_value=("value", "mean"),
        std_value=("value", "std"),
        total_duration=("duration", "sum"),
        mean_trial_duration=("duration", "mean"),
    )
    .reset_index()
)

# El CSV no conserva el tipo timedelta: se convierte aquí a segundos (float) para que
# los notebooks siguientes puedan leerlo y agregarlo (.mean(), .sum()...) sin problemas.
study_summary["total_duration"] = study_summary["total_duration"].dt.total_seconds()
study_summary["mean_trial_duration"] = study_summary["mean_trial_duration"].dt.total_seconds()

study_summary.head()

,sampler,problem,n_trials,best_value,mean_value,std_value,total_duration,mean_trial_duration
0,cmaes,zcat1,50,0.254027,0.153123,0.089824,4518.993983,90.379879
1,cmaes,zcat10,50,0.112726,0.068460,0.040746,5614.772873,112.295457
2,cmaes,zcat11,50,0.210841,0.117430,0.077371,4183.296339,83.665926
3,cmaes,zcat12,50,0.195056,0.127362,0.072957,3571.041948,71.420838
4,cmaes,zcat13,50,0.249413,0.154064,0.093711,5403.319708,108.066394


In [9]:
study_summary.to_csv(OUTPUT_DIR / "study_summary.csv", index=False)
print(f"Guardado: {(OUTPUT_DIR / 'study_summary.csv').resolve()}")
print(f"Filas: {len(study_summary)}  (esperado: {len(SAMPLERS) * len(PROBLEMS)})")

Guardado: D:\tfm\tfm\data\processed\study_summary.csv
Filas: 140  (esperado: 140)


## Tipos de datos del dataset maestro

In [10]:
master_df.dtypes

sampler                                               str
problem                                               str
number                                              int64
value                                             float64
state                                                 str
duration                                  timedelta64[us]
datetime_start                             datetime64[us]
datetime_complete                          datetime64[us]
params_archive_prune_policy                           str
params_archive_unbounded                             bool
params_blxab_alpha                                float64
params_blxab_beta                                 float64
params_cauchy_gamma                               float64
params_crossover                                      str
params_crossover_alpha                            float64
params_crossover_eta                              float64
params_crossover_prob                             float64
params_fuzzy_d

## Resumen de ficheros generados

- **`master_results.csv`** — un *trial* por fila, con todos los hiperparámetros, timestamps y duración. Fuente de `04_convergencia.ipynb`.
- **`study_summary.csv`** — un estudio por fila, con HV final, medias/desviaciones y duración total/media **en segundos**. Punto de partida de `02_ranking_global.ipynb`.

A partir de aquí, el resto de notebooks del TFM parten siempre de estos dos ficheros y no vuelven a tocar la base SQLite directamente (salvo el análisis de hiperparámetros con fANOVA, que sí necesita el objeto `Study` completo).
